# arcrl experiments (Kaggle T4, offline)

Not the competition submission. Attach: the `arcrl-code` dataset (`scripts/package_kaggle_dataset.sh`) and the ARC Prize 2026 ARC-AGI-3 competition. Set `CONFIGS` / `SPLIT`, run all, download `outputs.zip` from the output tab and unzip into the repo's `outputs/`.

In [ ]:
CONFIGS = ['configs/random.yaml', 'configs/dqn_baseline.yaml', 'configs/ppo_baseline.yaml']
SPLIT = 'dev'
SEEDS = None      # e.g. '0,1,2'; None = config seeds
BUDGET = None     # None = protocol budget
RUN_PROBE = True  # timing probe first

In [ ]:
import glob, os, subprocess
DS = glob.glob('/kaggle/input/**/repo.zip', recursive=True)[0].rsplit('/', 1)[0]
os.chdir('/kaggle/working')
!rm -rf arcrl && mkdir arcrl && cd arcrl && unzip -q {DS}/repo.zip && unzip -q {DS}/environment_files.zip
os.chdir('/kaggle/working/arcrl')
!pip install -q --no-index --find-links /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels arc-agi python-dotenv
!pip install -q --no-deps -e .
os.environ.update(OPERATION_MODE='offline', ENVIRONMENTS_DIR='/kaggle/working/arcrl/environment_files', MPLBACKEND='agg')
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -m pytest -q

In [ ]:
if RUN_PROBE:
    !python scripts/time_probe.py --actions 300

In [ ]:
for cfg in CONFIGS:
    args = ['python', 'scripts/evaluate.py', '--config', cfg, '--split', SPLIT]
    if SEEDS: args += ['--seeds', SEEDS]
    if BUDGET: args += ['--budget', str(BUDGET)]
    print('===', ' '.join(args)); subprocess.run(args, check=True)

In [ ]:
!python scripts/aggregate.py
!cat outputs/figures/summary.md
!cd /kaggle/working/arcrl && zip -qr /kaggle/working/outputs.zip outputs && ls -lh /kaggle/working/outputs.zip